In [10]:

# ============================================================
# 📦 INSTALLS
# ============================================================

!pip install -q openai chromadb pypdf tiktoken ipywidgets

# ============================================================
# 📚 IMPORTS
# ============================================================

import uuid
import textwrap

# from google.colab import files
# from google.colab import userdata

from openai import OpenAI
from pypdf import PdfReader

import chromadb

# ============================================================
# 🔑 OPENAI CLIENT
# ============================================================

# OPENAI_API_KEY = userdata.get("OPENAI_API_KEY")

# client = OpenAI(
#     api_key=OPENAI_API_KEY
# )

try:
    from dotenv import load_dotenv
    load_dotenv(Path('.env'))
except Exception:
    pass

import requests

BCAI_PAT = os.getenv('BCAI_PAT', '')

if not BCAI_PAT:
    raise ValueError('BCAI_PAT is missing. Set it in .env before continuing.')

url = 'https://bcai-test.web.boeing.com/bcai-public-api/embedding'

headers = {
    'accept': 'application/json',
    'Content-Type': 'application/json',
    'Authorization': f'basic {BCAI_PAT}'
}
# ============================================================
# 📄 LOAD PDF
# ============================================================

def load_pdf(pdf_path):

    reader = PdfReader(pdf_path)

    text = ""

    for page in reader.pages:

        extracted = page.extract_text()

        if extracted:
            text += extracted + "\n"

    return text

# ============================================================
# ✂️ CHUNKER WITH OVERLAP
# ============================================================

def chunk_text(
    text,
    chunk_size=800,
    overlap=200
):

    chunks = []

    start = 0

    while start < len(text):

        end = start + chunk_size

        chunk = text[start:end]

        chunks.append(chunk)

        # move forward with overlap
        start += chunk_size - overlap

    return chunks

# ============================================================
# 🧠 CREATE EMBEDDING
# ============================================================

    
# def get_embedding(text):

#     response = client.embeddings.create(
#         model="text-embedding-3-small",
#         input=text
#     )

def get_embedding(text, model='text-embedding-3-small'):
    """Get embedding vector for a single text string."""
    data = {'input': text, 'model': model}
    response = requests.post(url, headers=headers, json=data, timeout=60)
    result = response.json()

    return result['data'][0]['embedding']

# ============================================================
# 🗂️ CHROMADB SETUP
# ============================================================

chroma_client = chromadb.Client()

collection = chroma_client.get_or_create_collection(
    name="simple_rag"
)

# ============================================================
# 📥 STORE CHUNKS
# ============================================================

def add_chunks_to_chroma(chunks):

    for chunk in chunks:

        embedding = get_embedding(chunk)

        collection.add(
            ids=[str(uuid.uuid4())],
            documents=[chunk],
            embeddings=[embedding]
        )

    print(f"Stored {len(chunks)} chunks in ChromaDB.")

# ============================================================
# 🔎 RETRIEVE DOCUMENTS
# ============================================================

def retrieve(query, top_k=3):

    query_embedding = get_embedding(query)

    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=top_k
    )

    docs = results["documents"][0]

    return docs

# ============================================================
# 🤖 ASK QUESTION
# ============================================================

OPENAI_API_BASE = 'https://bcai-openai-proxy-test.taspre-phx.apps.boeing.com/v1'
OPENAI_MODEL = 'gpt-4.1-mini'

chat_client = OpenAI(base_url=OPENAI_API_BASE, api_key=BCAI_PAT, timeout=60.0)
    
def ask(query):

    docs = retrieve(query)

    context = "\n\n".join(docs)

    prompt = f"""
You are a helpful RAG assistant.

Answer ONLY from the provided context.

If the answer is not in the context, say:
"I could not find the answer in the document."

Context:
{context}

Question:
{query}
"""

    # response = client.chat.completions.create(
    #     model="gpt-4o-mini",
    #     messages=[
    #         {
    #             "role": "system",
    #             "content": "You answer questions using retrieved documents."
    #         },
    #         {
    #             "role": "user",
    #             "content": prompt
    #         }
    #     ],
    #     temperature=0.2
    # )

    response = chat_client.chat.completions.create(
        model=OPENAI_MODEL,
        messages=[
                {
                    "role": "system",
                    "content": "You answer questions using retrieved documents."
                },
                {
                    "role": "user",
                    "content": prompt
                }
            ],
        max_tokens=200,
        temperature=0.2
    )
    # result = response.json()
    return response.choices[0].message.content

# ============================================================
# 📂 UPLOAD PDF
# ============================================================

# print("Upload your PDF file.\n")

# uploaded = files.upload()

# pdf_path = list(uploaded.keys())[0]

# print(f"\nLoaded PDF: {pdf_path}")

pdf_path = r"C:\Users\yf297e\Downloads\VISTA_Contracts_2026-05-11_13-09-08.pdf"
print(f"Loaded PDF: {pdf_path}")


# ============================================================
# 🏗️ BUILD RAG PIPELINE
# ============================================================

print("\nLoading PDF...")

text = load_pdf(pdf_path)

print("PDF loaded.")

print("\nChunking document...")

chunks = chunk_text(
    text,
    chunk_size=800,
    overlap=200
)

print(f"Created {len(chunks)} chunks.")

print("\nGenerating embeddings and storing in ChromaDB...")

add_chunks_to_chroma(chunks)

print("\n✅ RAG Pipeline Ready")

# ============================================================
# 💬 INTERACTIVE CHATBOT
# ============================================================

print("\n🤖 Chatbot Ready")
print("Type 'exit' to stop.\n")

while True:

    query = input("You: ")

    if query.lower() == "exit":

        print("\nBot: Goodbye.")
        break

    answer = ask(query)

    print("\nBot:")

    print(textwrap.fill(answer, width=100))

    print("\n" + "=" * 100 + "\n")

Loaded PDF: C:\Users\yf297e\Downloads\VISTA_Contracts_2026-05-11_13-09-08.pdf

Loading PDF...
PDF loaded.

Chunking document...
Created 7 chunks.

Generating embeddings and storing in ChromaDB...
Stored 7 chunks in ChromaDB.

✅ RAG Pipeline Ready

🤖 Chatbot Ready
Type 'exit' to stop.



You:  WHat is te=he title of the document?



Bot:
I could not find the answer in the document.




You:  no of pages?



Bot:
I could not find the answer in the document.




You:  What is the contract Name for the contract number CN-00-002?



Bot:
The contract Name for the contract number CN-00-002 is "Flight maintenance."




You:  exit



Bot: Goodbye.
